In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from collections import defaultdict

# ============================================
# CONFIG
# ============================================

data_names = [
    'sarscov2_record',
]

# -------- DISPLAY TITLES --------
title_map = {
    "sarscov2_record": "SARS-CoV-2",
}

# -------- VISUAL CONTROL --------
FONT_X = 14
FONT_Y = 14
FONT_TITLE = 15
FONT_LEGEND = 11

GRID_ON = True
SPINES_ON = True

OUTDIR = "k_selection_clean"
os.makedirs(OUTDIR, exist_ok=True)

# ============================================
# NATURE STYLE
# ============================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
})

COLORS = {
    "coverage": "#4C72B0",
    "info_sparsity": "#55A868",
    "stability_entropy": "#C44E52",
}

# ============================================
# METRICS
# ============================================

def coverage(X):
    return np.mean((X > 0).sum(axis=1) / X.shape[1])

def entropy(X):
    row_sums = X.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    P = X / row_sums
    P_safe = np.where(P > 0, P, 1)
    return -np.mean(np.sum(P * np.log(P_safe), axis=1))

def rare_fraction(X):
    rare = (X == 1).sum(axis=1)
    nonzero = (X > 0).sum(axis=1)
    nonzero = np.where(nonzero == 0, 1, nonzero)
    return np.mean(rare / nonzero)

def info_sparsity(X):
    cov = coverage(X)
    return entropy(X) / (1 - cov + 1e-12)

def stability_entropy(X):
    return entropy(X) * (1 - rare_fraction(X))

# ============================================
# HELPERS
# ============================================

def detect_elbow(ks, values):
    values = np.asarray(values)
    if len(values) < 3:
        return None
    second_diff = np.diff(values, n=2)
    return ks[np.argmin(second_diff) + 1]

def pick_peak_k(ks, values, mode="max"):
    ks = list(ks)
    values = np.asarray(values)

    if mode == "max":
        return ks[np.argmax(values)]
    else:
        return ks[np.argmin(values)]

def normalize(values):
    values = np.asarray(values)
    vmin, vmax = values.min(), values.max()
    if vmax - vmin < 1e-12:
        return np.zeros_like(values)
    return (values - vmin) / (vmax - vmin)

# ============================================
# LEGEND EXPORT
# ============================================

def save_legend(handles, labels):

    fig = plt.figure(figsize=(8, 1))
    fig.legend(handles, labels, loc='center', ncol=len(labels), fontsize=FONT_LEGEND)
    fig.savefig(os.path.join(OUTDIR, "legend_horizontal.pdf"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)

    fig = plt.figure(figsize=(3, 4))
    fig.legend(handles, labels, loc='center', ncol=1, fontsize=FONT_LEGEND)
    fig.savefig(os.path.join(OUTDIR, "legend_vertical.pdf"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)

# ============================================
# STEP 1: COLLECT DATA + BEST k
# ============================================

all_best_k = {
    "coverage_elbow": [],
    "info_sparsity": [],
    "stability_entropy": [],
}

dataset_results = []

for data_name in data_names:

    print("\n==============================")
    print(f"DATASET: {data_name}")
    print("==============================")

    base_dir = f"features2/CAKR/{data_name}"


    k_values = sorted([
        int(re.match(r"k(\d+)_facet0\.npy", f).group(1))
        for f in os.listdir(base_dir)
        if re.match(r"k(\d+)_facet0\.npy", f)
    ])


    results = {
        "coverage": {},
        "info_sparsity": {},
        "stability_entropy": {},
    }

    for k in k_values:
        X = np.load(os.path.join(base_dir, f"k{k}_facet0.npy"))

        results["coverage"][k] = coverage(X)
        results["info_sparsity"][k] = info_sparsity(X)
        results["stability_entropy"][k] = stability_entropy(X)

    ks = k_values
    cov_vals = [results["coverage"][k] for k in ks]
    info_vals = [results["info_sparsity"][k] for k in ks]
    stab_vals = [results["stability_entropy"][k] for k in ks]

    elbow_k = detect_elbow(ks, cov_vals)

    best_k = {
        "coverage_elbow": elbow_k,
        "info_sparsity": pick_peak_k(ks, info_vals, "max"),
        "stability_entropy": pick_peak_k(ks, stab_vals, "max"),
    }

    for m in best_k:
        all_best_k[m].append(best_k[m])

    dataset_results.append((data_name, ks, cov_vals, info_vals, stab_vals, best_k))

# ============================================
# STEP 2: COMPUTE S(m)
# ============================================

lambda_param = 0.5
S_scores = {}

for m in ["coverage_elbow", "info_sparsity", "stability_entropy"]:

    diffs = []
    abs_diffs = []

    for j in range(len(data_names)):
        k_cov = all_best_k["coverage_elbow"][j]
        k_m = all_best_k[m][j]

        diffs.append(k_cov - k_m)
        abs_diffs.append(abs(k_cov - k_m))

    S_scores[m] = np.mean(diffs) - lambda_param * np.mean(abs_diffs)

print("\n==============================")
print("S(m) scores")
print("==============================")
for m in S_scores:
    print(f"{m}: {S_scores[m]:.4f}")

# ============================================
# STEP 3: RANK → WEIGHTS
# ============================================

sorted_metrics = sorted(S_scores, key=S_scores.get)

metric_weights = {}
for rank, m in enumerate(sorted_metrics, start=1):
    metric_weights[m] = rank

print("\nWeights from ranking:")
print(metric_weights)

# ============================================
# STEP 4: GLOBAL VOTING ACROSS ALL DATASETS
# ============================================

global_votes = defaultdict(float)

for (data_name, ks, cov_vals, info_vals, stab_vals, best_k) in dataset_results:
    for m, k in best_k.items():
        global_votes[k] += metric_weights[m]

final_global_k = max(global_votes, key=global_votes.get)

# ============================================
# STEP 5: FINAL LOOP (PLOTS USE GLOBAL k)
# ============================================

for (data_name, ks, cov_vals, info_vals, stab_vals, best_k) in dataset_results:

    print("\n==============================")
    print(f"DATASET: {data_name}")
    print("==============================")

    print("Suggested k:")
    for m in best_k:
        print(f"  {m}: {best_k[m]}")
    print(f"Global selected k: {final_global_k}")

    # ============================================
    # PLOT
    # ============================================

    fig, ax = plt.subplots(figsize=(6.5, 4.2))

    l1, = ax.plot(ks, normalize(cov_vals), marker='o', linewidth=1.8,
                  color=COLORS["coverage"], label="Coverage")

    l2, = ax.plot(ks, normalize(info_vals), marker='o', linewidth=1.8,
                  color=COLORS["info_sparsity"], label="Information–sparsity")

    l3, = ax.plot(ks, normalize(stab_vals), marker='o', linewidth=1.8,
                  color=COLORS["stability_entropy"], label="Stability-adjusted entropy")

    v1 = None
    if best_k["coverage_elbow"] is not None:
        v1 = ax.axvline(best_k["coverage_elbow"], linestyle='--', linewidth=1.5,
                        color='gray', label='Coverage elbow')

    v2 = ax.axvline(final_global_k, linestyle=':', linewidth=1.5,
                    color='black', label='Selected k')

    ax.set_xlabel("k", fontsize=FONT_X)
    ax.set_ylabel("Normalized score", fontsize=FONT_Y)
    ax.set_title(title_map.get(data_name, data_name.replace("_", " ")), fontsize=FONT_TITLE)

    if GRID_ON:
        ax.grid(alpha=0.15)

    if not SPINES_ON:
        for spine in ax.spines.values():
            spine.set_visible(False)
    else:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    handles = [l1, l2, l3]
    labels = ["Coverage", "Information–sparsity", "Stability-adjusted entropy"]

    if v1 is not None:
        handles.append(v1)
        labels.append("Coverage elbow")

    handles.append(v2)
    labels.append("Selected k")

    ax.legend(handles, labels,
              loc='upper center',
              bbox_to_anchor=(0.5, -0.28),
              ncol=3,
              fontsize=FONT_LEGEND,
              frameon=False)

    plt.tight_layout()

    plt.savefig(os.path.join(OUTDIR, f"{data_name}_nature.pdf"),
                dpi=300, bbox_inches='tight')

    plt.close()

    save_legend(handles, labels)

# ============================================
# GLOBAL RESULT
# ============================================

print("\n==============================")
print("GLOBAL FINAL k")
print("==============================")
print(final_global_k)

print("\n==============================")
print("GLOBAL VOTE TOTALS")
print("==============================")
for k in sorted(global_votes):
    print(f"k={k}: {global_votes[k]}")


DATASET: sarscov2_record

S(m) scores
coverage_elbow: 0.0000
info_sparsity: 1.0000
stability_entropy: 0.0000

Weights from ranking:
{'coverage_elbow': 1, 'stability_entropy': 2, 'info_sparsity': 3}

DATASET: sarscov2_record
Suggested k:
  coverage_elbow: 6
  info_sparsity: 4
  stability_entropy: 6
Global selected k: 6

GLOBAL FINAL k
6

GLOBAL VOTE TOTALS
k=4: 3.0
k=6: 3.0


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from collections import defaultdict

# ============================================
# CONFIG
# ============================================

data_names = [
    'rhinovirus_record',
]

# -------- DISPLAY TITLES --------
title_map = {
    "rhinovirus_record": "rhinovirus",
}

# -------- VISUAL CONTROL --------
FONT_X = 14
FONT_Y = 14
FONT_TITLE = 15
FONT_LEGEND = 11

GRID_ON = True
SPINES_ON = True

OUTDIR = "k_selection_clean"
os.makedirs(OUTDIR, exist_ok=True)

# ============================================
# NATURE STYLE
# ============================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
})

COLORS = {
    "coverage": "#4C72B0",
    "info_sparsity": "#55A868",
    "stability_entropy": "#C44E52",
}

# ============================================
# METRICS
# ============================================

def coverage(X):
    return np.mean((X > 0).sum(axis=1) / X.shape[1])

def entropy(X):
    row_sums = X.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    P = X / row_sums
    P_safe = np.where(P > 0, P, 1)
    return -np.mean(np.sum(P * np.log(P_safe), axis=1))

def rare_fraction(X):
    rare = (X == 1).sum(axis=1)
    nonzero = (X > 0).sum(axis=1)
    nonzero = np.where(nonzero == 0, 1, nonzero)
    return np.mean(rare / nonzero)

def info_sparsity(X):
    cov = coverage(X)
    return entropy(X) / (1 - cov + 1e-12)

def stability_entropy(X):
    return entropy(X) * (1 - rare_fraction(X))

# ============================================
# HELPERS
# ============================================

def detect_elbow(ks, values):
    values = np.asarray(values)
    if len(values) < 3:
        return None
    second_diff = np.diff(values, n=2)
    return ks[np.argmin(second_diff) + 1]

def pick_peak_k(ks, values, mode="max"):
    ks = list(ks)
    values = np.asarray(values)

    if mode == "max":
        return ks[np.argmax(values)]
    else:
        return ks[np.argmin(values)]

def normalize(values):
    values = np.asarray(values)
    vmin, vmax = values.min(), values.max()
    if vmax - vmin < 1e-12:
        return np.zeros_like(values)
    return (values - vmin) / (vmax - vmin)

# ============================================
# LEGEND EXPORT
# ============================================

def save_legend(handles, labels):

    fig = plt.figure(figsize=(8, 1))
    fig.legend(handles, labels, loc='center', ncol=len(labels), fontsize=FONT_LEGEND)
    fig.savefig(os.path.join(OUTDIR, "legend_horizontal.pdf"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)

    fig = plt.figure(figsize=(3, 4))
    fig.legend(handles, labels, loc='center', ncol=1, fontsize=FONT_LEGEND)
    fig.savefig(os.path.join(OUTDIR, "legend_vertical.pdf"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)

# ============================================
# STEP 1: COLLECT DATA + BEST k
# ============================================

all_best_k = {
    "coverage_elbow": [],
    "info_sparsity": [],
    "stability_entropy": [],
}

dataset_results = []

for data_name in data_names:

    print("\n==============================")
    print(f"DATASET: {data_name}")
    print("==============================")

    base_dir = f"features2/CAKR/{data_name}"


    k_values = sorted([
        int(re.match(r"k(\d+)_facet0\.npy", f).group(1))
        for f in os.listdir(base_dir)
        if re.match(r"k(\d+)_facet0\.npy", f)
    ])


    results = {
        "coverage": {},
        "info_sparsity": {},
        "stability_entropy": {},
    }

    for k in k_values:
        X = np.load(os.path.join(base_dir, f"k{k}_facet0.npy"))

        results["coverage"][k] = coverage(X)
        results["info_sparsity"][k] = info_sparsity(X)
        results["stability_entropy"][k] = stability_entropy(X)

    ks = k_values
    cov_vals = [results["coverage"][k] for k in ks]
    info_vals = [results["info_sparsity"][k] for k in ks]
    stab_vals = [results["stability_entropy"][k] for k in ks]

    elbow_k = detect_elbow(ks, cov_vals)

    best_k = {
        "coverage_elbow": elbow_k,
        "info_sparsity": pick_peak_k(ks, info_vals, "max"),
        "stability_entropy": pick_peak_k(ks, stab_vals, "max"),
    }

    for m in best_k:
        all_best_k[m].append(best_k[m])

    dataset_results.append((data_name, ks, cov_vals, info_vals, stab_vals, best_k))

# ============================================
# STEP 2: COMPUTE S(m)
# ============================================

lambda_param = 0.5
S_scores = {}

for m in ["coverage_elbow", "info_sparsity", "stability_entropy"]:

    diffs = []
    abs_diffs = []

    for j in range(len(data_names)):
        k_cov = all_best_k["coverage_elbow"][j]
        k_m = all_best_k[m][j]

        diffs.append(k_cov - k_m)
        abs_diffs.append(abs(k_cov - k_m))

    S_scores[m] = np.mean(diffs) - lambda_param * np.mean(abs_diffs)

print("\n==============================")
print("S(m) scores")
print("==============================")
for m in S_scores:
    print(f"{m}: {S_scores[m]:.4f}")

# ============================================
# STEP 3: RANK → WEIGHTS
# ============================================

sorted_metrics = sorted(S_scores, key=S_scores.get)

metric_weights = {}
for rank, m in enumerate(sorted_metrics, start=1):
    metric_weights[m] = rank

print("\nWeights from ranking:")
print(metric_weights)

# ============================================
# STEP 4: GLOBAL VOTING ACROSS ALL DATASETS
# ============================================

global_votes = defaultdict(float)

for (data_name, ks, cov_vals, info_vals, stab_vals, best_k) in dataset_results:
    for m, k in best_k.items():
        global_votes[k] += metric_weights[m]

final_global_k = max(global_votes, key=global_votes.get)

# ============================================
# STEP 5: FINAL LOOP (PLOTS USE GLOBAL k)
# ============================================

for (data_name, ks, cov_vals, info_vals, stab_vals, best_k) in dataset_results:

    print("\n==============================")
    print(f"DATASET: {data_name}")
    print("==============================")

    print("Suggested k:")
    for m in best_k:
        print(f"  {m}: {best_k[m]}")
    print(f"Global selected k: {final_global_k}")

    # ============================================
    # PLOT
    # ============================================

    fig, ax = plt.subplots(figsize=(6.5, 4.2))

    l1, = ax.plot(ks, normalize(cov_vals), marker='o', linewidth=1.8,
                  color=COLORS["coverage"], label="Coverage")

    l2, = ax.plot(ks, normalize(info_vals), marker='o', linewidth=1.8,
                  color=COLORS["info_sparsity"], label="Information–sparsity")

    l3, = ax.plot(ks, normalize(stab_vals), marker='o', linewidth=1.8,
                  color=COLORS["stability_entropy"], label="Stability-adjusted entropy")

    v1 = None
    if best_k["coverage_elbow"] is not None:
        v1 = ax.axvline(best_k["coverage_elbow"], linestyle='--', linewidth=1.5,
                        color='gray', label='Coverage elbow')

    v2 = ax.axvline(final_global_k, linestyle=':', linewidth=1.5,
                    color='black', label='Selected k')

    ax.set_xlabel("k", fontsize=FONT_X)
    ax.set_ylabel("Normalized score", fontsize=FONT_Y)
    ax.set_title(title_map.get(data_name, data_name.replace("_", " ")), fontsize=FONT_TITLE)

    if GRID_ON:
        ax.grid(alpha=0.15)

    if not SPINES_ON:
        for spine in ax.spines.values():
            spine.set_visible(False)
    else:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    handles = [l1, l2, l3]
    labels = ["Coverage", "Information–sparsity", "Stability-adjusted entropy"]

    if v1 is not None:
        handles.append(v1)
        labels.append("Coverage elbow")

    handles.append(v2)
    labels.append("Selected k")

    ax.legend(handles, labels,
              loc='upper center',
              bbox_to_anchor=(0.5, -0.28),
              ncol=3,
              fontsize=FONT_LEGEND,
              frameon=False)

    plt.tight_layout()

    plt.savefig(os.path.join(OUTDIR, f"{data_name}_nature.pdf"),
                dpi=300, bbox_inches='tight')

    plt.close()

    save_legend(handles, labels)

# ============================================
# GLOBAL RESULT
# ============================================

print("\n==============================")
print("GLOBAL FINAL k")
print("==============================")
print(final_global_k)

print("\n==============================")
print("GLOBAL VOTE TOTALS")
print("==============================")
for k in sorted(global_votes):
    print(f"k={k}: {global_votes[k]}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from collections import defaultdict

# ============================================
# CONFIG
# ============================================

data_names = [
    'ebolavirus_record',
    'HEV_record',
    'influenzaHAgene_record',
    'mammalianMT_record',
]

# -------- DISPLAY TITLES --------
title_map = {
    "mammalianMT_record": "MammalianMT",
    "HEV_record": "HEV",
    "influenzaHAgene_record": "influenzaHAgene",
    "ebolavirus_record": "ebolavirus",
}

# -------- VISUAL CONTROL --------
FONT_X = 14
FONT_Y = 14
FONT_TITLE = 15
FONT_LEGEND = 11

GRID_ON = True
SPINES_ON = True

OUTDIR = "k_selection_clean"
os.makedirs(OUTDIR, exist_ok=True)

# ============================================
# NATURE STYLE
# ============================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
})

COLORS = {
    "coverage": "#4C72B0",
    "info_sparsity": "#55A868",
    "stability_entropy": "#C44E52",
}

# ============================================
# METRICS
# ============================================

def coverage(X):
    return np.mean((X > 0).sum(axis=1) / X.shape[1])

def entropy(X):
    row_sums = X.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    P = X / row_sums
    P_safe = np.where(P > 0, P, 1)
    return -np.mean(np.sum(P * np.log(P_safe), axis=1))

def rare_fraction(X):
    rare = (X == 1).sum(axis=1)
    nonzero = (X > 0).sum(axis=1)
    nonzero = np.where(nonzero == 0, 1, nonzero)
    return np.mean(rare / nonzero)

def info_sparsity(X):
    cov = coverage(X)
    return entropy(X) / (1 - cov + 1e-12)

def stability_entropy(X):
    return entropy(X) * (1 - rare_fraction(X))

# ============================================
# HELPERS
# ============================================

def detect_elbow(ks, values):
    values = np.asarray(values)
    if len(values) < 3:
        return None
    second_diff = np.diff(values, n=2)
    return ks[np.argmin(second_diff) + 1]

def pick_peak_k(ks, values, mode="max"):
    ks = list(ks)
    values = np.asarray(values)

    if mode == "max":
        return ks[np.argmax(values)]
    else:
        return ks[np.argmin(values)]

def normalize(values):
    values = np.asarray(values)
    vmin, vmax = values.min(), values.max()
    if vmax - vmin < 1e-12:
        return np.zeros_like(values)
    return (values - vmin) / (vmax - vmin)

# ============================================
# LEGEND EXPORT
# ============================================

def save_legend(handles, labels):

    fig = plt.figure(figsize=(8, 1))
    fig.legend(handles, labels, loc='center', ncol=len(labels), fontsize=FONT_LEGEND)
    fig.savefig(os.path.join(OUTDIR, "legend_horizontal.pdf"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)

    fig = plt.figure(figsize=(3, 4))
    fig.legend(handles, labels, loc='center', ncol=1, fontsize=FONT_LEGEND)
    fig.savefig(os.path.join(OUTDIR, "legend_vertical.pdf"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)

# ============================================
# STEP 1: COLLECT DATA + BEST k
# ============================================

all_best_k = {
    "coverage_elbow": [],
    "info_sparsity": [],
    "stability_entropy": [],
}

dataset_results = []

for data_name in data_names:

    print("\n==============================")
    print(f"DATASET: {data_name}")
    print("==============================")

    base_dir = f"features2/CAKR/{data_name}"


    k_values = sorted([
        int(re.match(r"k(\d+)_facet0\.npy", f).group(1))
        for f in os.listdir(base_dir)
        if re.match(r"k(\d+)_facet0\.npy", f)
    ])


    results = {
        "coverage": {},
        "info_sparsity": {},
        "stability_entropy": {},
    }

    for k in k_values:
        X = np.load(os.path.join(base_dir, f"k{k}_facet0.npy"))

        results["coverage"][k] = coverage(X)
        results["info_sparsity"][k] = info_sparsity(X)
        results["stability_entropy"][k] = stability_entropy(X)

    ks = k_values
    cov_vals = [results["coverage"][k] for k in ks]
    info_vals = [results["info_sparsity"][k] for k in ks]
    stab_vals = [results["stability_entropy"][k] for k in ks]

    elbow_k = detect_elbow(ks, cov_vals)

    best_k = {
        "coverage_elbow": elbow_k,
        "info_sparsity": pick_peak_k(ks, info_vals, "max"),
        "stability_entropy": pick_peak_k(ks, stab_vals, "max"),
    }

    for m in best_k:
        all_best_k[m].append(best_k[m])

    dataset_results.append((data_name, ks, cov_vals, info_vals, stab_vals, best_k))

# ============================================
# STEP 2: COMPUTE S(m)
# ============================================

lambda_param = 0.5
S_scores = {}

for m in ["coverage_elbow", "info_sparsity", "stability_entropy"]:

    diffs = []
    abs_diffs = []

    for j in range(len(data_names)):
        k_cov = all_best_k["coverage_elbow"][j]
        k_m = all_best_k[m][j]

        diffs.append(k_cov - k_m)
        abs_diffs.append(abs(k_cov - k_m))

    S_scores[m] = np.mean(diffs) - lambda_param * np.mean(abs_diffs)

print("\n==============================")
print("S(m) scores")
print("==============================")
for m in S_scores:
    print(f"{m}: {S_scores[m]:.4f}")

# ============================================
# STEP 3: RANK → WEIGHTS
# ============================================

sorted_metrics = sorted(S_scores, key=S_scores.get)

metric_weights = {}
for rank, m in enumerate(sorted_metrics, start=1):
    metric_weights[m] = rank

print("\nWeights from ranking:")
print(metric_weights)

# ============================================
# STEP 4: GLOBAL VOTING ACROSS ALL DATASETS
# ============================================

global_votes = defaultdict(float)

for (data_name, ks, cov_vals, info_vals, stab_vals, best_k) in dataset_results:
    for m, k in best_k.items():
        global_votes[k] += metric_weights[m]

final_global_k = max(global_votes, key=global_votes.get)

# ============================================
# STEP 5: FINAL LOOP (PLOTS USE GLOBAL k)
# ============================================

for (data_name, ks, cov_vals, info_vals, stab_vals, best_k) in dataset_results:

    print("\n==============================")
    print(f"DATASET: {data_name}")
    print("==============================")

    print("Suggested k:")
    for m in best_k:
        print(f"  {m}: {best_k[m]}")
    print(f"Global selected k: {final_global_k}")

    # ============================================
    # PLOT
    # ============================================

    fig, ax = plt.subplots(figsize=(6.5, 4.2))

    l1, = ax.plot(ks, normalize(cov_vals), marker='o', linewidth=1.8,
                  color=COLORS["coverage"], label="Coverage")

    l2, = ax.plot(ks, normalize(info_vals), marker='o', linewidth=1.8,
                  color=COLORS["info_sparsity"], label="Information–sparsity")

    l3, = ax.plot(ks, normalize(stab_vals), marker='o', linewidth=1.8,
                  color=COLORS["stability_entropy"], label="Stability-adjusted entropy")

    v1 = None
    if best_k["coverage_elbow"] is not None:
        v1 = ax.axvline(best_k["coverage_elbow"], linestyle='--', linewidth=1.5,
                        color='gray', label='Coverage elbow')

    v2 = ax.axvline(final_global_k, linestyle=':', linewidth=1.5,
                    color='black', label='Selected k')

    ax.set_xlabel("k", fontsize=FONT_X)
    ax.set_ylabel("Normalized score", fontsize=FONT_Y)
    ax.set_title(title_map.get(data_name, data_name.replace("_", " ")), fontsize=FONT_TITLE)

    if GRID_ON:
        ax.grid(alpha=0.15)

    if not SPINES_ON:
        for spine in ax.spines.values():
            spine.set_visible(False)
    else:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    handles = [l1, l2, l3]
    labels = ["Coverage", "Information–sparsity", "Stability-adjusted entropy"]

    if v1 is not None:
        handles.append(v1)
        labels.append("Coverage elbow")

    handles.append(v2)
    labels.append("Selected k")

    ax.legend(handles, labels,
              loc='upper center',
              bbox_to_anchor=(0.5, -0.28),
              ncol=3,
              fontsize=FONT_LEGEND,
              frameon=False)

    plt.tight_layout()

    plt.savefig(os.path.join(OUTDIR, f"{data_name}_nature.pdf"),
                dpi=300, bbox_inches='tight')

    plt.close()

    save_legend(handles, labels)

# ============================================
# GLOBAL RESULT
# ============================================

print("\n==============================")
print("GLOBAL FINAL k")
print("==============================")
print(final_global_k)

print("\n==============================")
print("GLOBAL VOTE TOTALS")
print("==============================")
for k in sorted(global_votes):
    print(f"k={k}: {global_votes[k]}")